# 260527 ML Training

| 항목 | 내용 |
|------|------|
| 데이터 | `outputs/260526_preprocessing/` |
| 예측인자 | VIF 생존 피처 (`vif_survived.csv`) |
| 결측 처리 | KNN / Mean / Median / Most-Frequent / Constant (노트북에서 선택) |
| CV | Nested 5-Fold |
| 모델 | SVM · Logistic Regression · Random Forest · XGBoost · LightGBM · Gradient Boosting · Extra Trees |
| 평가 지표 | Accuracy / Sensitivity / Specificity / Precision / F1 / AUC |
| 저장 | Confusion Matrix, ROC Curve, CSV |

---

## Nested 5-Fold CV

```
전체 데이터
└── Outer 5-Fold
     └── [Fold 1~5 반복]
          ├── Outer Train (80%)
          │    └── Inner 5-Fold (GridSearchCV, scoring=AUC)
          │         └── 최적 하이퍼파라미터 선택
          │              └── Outer Train 전체로 재학습  ← refit=True
          └── Outer Test (20%)
               └── 재학습된 모델로 평가 → mean ± std 보고
```

> KNN Imputer / StandardScaler 모두 Outer Train fit → Outer Test transform (data leakage 방지)


In [1]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path().resolve()
ROOT = NOTEBOOK_DIR.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from core.ml_trainer import FeatureInspector, MLConfig, MLPipeline

## 공통 설정

아래 셀에서 **KNN k 값**, **모델 목록** 등을 수정할 수 있다.

In [2]:
# ── 수정 가능한 옵션 ──────────────────────────────────────────────
N_OUTER_FOLDS   = 5          # Outer CV fold 수
N_INNER_FOLDS   = 5          # Inner CV fold 수 (하이퍼파라미터 탐색)
RANDOM_STATE    = 42
MODELS          = ["svm", "lr", "rf", "xgb", "lgbm", "gb", "et"]  # 학습할 모델 목록
SCALE_FEATURES  = True       # SVM·LR을 위한 StandardScaler 적용 여부

# ── 결측치 처리 전략 ──────────────────────────────────────────────
# "knn"          : KNNImputer — k개 최근접 이웃 평균으로 대체 (KNN_N_NEIGHBORS 사용)
# "mean"         : 각 피처의 평균으로 대체
# "median"       : 각 피처의 중앙값으로 대체
# "most_frequent": 각 피처의 최빈값으로 대체
# "constant"     : IMPUTATION_FILL_VALUE 값으로 일괄 대체
IMPUTATION_STRATEGY   = "mean"  # "knn" | "mean" | "median" | "most_frequent" | "constant"
KNN_N_NEIGHBORS       = 5      # IMPUTATION_STRATEGY="knn"일 때 사용할 k 값
IMPUTATION_FILL_VALUE = 0.0    # IMPUTATION_STRATEGY="constant"일 때 사용할 대체 값

# ── 하이퍼파라미터 탐색 방법 ──────────────────────────────────────
# "grid"  : GridSearchCV  — 격자 탐색 (빠르지만 탐색 범위 제한)
# "bayes" : BayesSearchCV — 베이지안 최적화 (느리지만 넓은 연속 공간 탐색)
TUNER        = "bayes"   # "grid" | "bayes"
N_BAYES_ITER = 30       # TUNER="bayes"일 때 모델당 탐색 횟수 (클수록 정확, 느림)

# ── 예측 인자 직접 지정 (None 이면 vif_survived.csv 자동 로드) ────
# 인코딩 후 컬럼명으로 지정한다. 아래 출력에서 복사하여 사용하면 된다.
DIABETES_FEATURES: list[str] | None = ['A/G ratio', 'AFP', 'ALP', 'Albumin', 'B/C ratio', 'BMI', 'BUN', 'Basophil', 'CPK', 'Creatinine', 'D.Bilirubin', 'Eosinophil', 'GOT(AST)', 'GPT(ALT)', 'Globulin', 'HDL', 'Hct', 'Hgb', 'I.Bilirubin', 'LDH', 'LDL', 'Lymphocyte', 'MCH', 'MCHC', 'MCV', 'MPV', 'Monocyte', 'PCT', 'PDW', 'PH', 'Platelet', 'RBC', 'RDW', 'SG', 'T.Bilirubin', 'T.Cholesterol', 'T.Protein', 'TSH', 'Triglyceride', 'Uric acid', 'WBC', 'e-GFR', 'r-GTP', '공복혈당', '나이', '백혈구(소변현미경)', '비만도', '신장', '적혈구(소변현미경)', '체중', '허리둘레', '혈압(수축기)', '혈압(이완기)', 'gender_여자', 'Bilirubin_음성', 'Blood_음성', 'Glucose_음성', 'Keton_음성', 'Leukocyte_음성', 'Nitrite_음성', 'Protein_음성', 'Urobilinogen_음성', 'HBs-Ag_음성', 'HBs-Ab_음성']
PREDIABETES_FEATURES: list[str] | None = ['A/G ratio', 'AFP', 'ALP', 'Albumin', 'B/C ratio', 'BMI', 'BUN', 'Basophil', 'CPK', 'Creatinine', 'D.Bilirubin', 'Eosinophil', 'GOT(AST)', 'GPT(ALT)', 'Globulin', 'HDL', 'Hct', 'Hgb', 'I.Bilirubin', 'LDH', 'LDL', 'Lymphocyte', 'MCH', 'MCHC', 'MCV', 'MPV', 'Monocyte', 'PCT', 'PDW', 'PH', 'Platelet', 'RBC', 'RDW', 'SG', 'T.Bilirubin', 'T.Cholesterol', 'T.Protein', 'TSH', 'Triglyceride', 'Uric acid', 'WBC', 'e-GFR', 'r-GTP', '공복혈당', '나이', '백혈구(소변현미경)', '비만도', '신장', '적혈구(소변현미경)', '체중', '허리둘레', '혈압(수축기)', '혈압(이완기)', 'gender_여자', 'Bilirubin_음성', 'Blood_음성', 'Glucose_음성', 'Keton_음성', 'Leukocyte_음성', 'Nitrite_음성', 'Protein_음성', 'Urobilinogen_음성', 'HBs-Ag_음성', 'HBs-Ab_음성']

# 예시 — 일부 피처만 사용하고 싶을 때:
# DIABETES_FEATURES = ['공복혈당', 'BMI', '나이', 'HDL', 'Triglyceride', 'gender_여자']
# ─────────────────────────────────────────────────────────────────

_ = FeatureInspector.show(ROOT / "outputs" / "260526_preprocessing" / "diabetes_dataset.xlsx",     title="Diabetes")
_ = FeatureInspector.show(ROOT / "outputs" / "260526_preprocessing" / "pre_diabetes_dataset.xlsx", title="Pre-diabetes")

── Diabetes (62개) ──
['A/G ratio', 'AFP', 'ALP', 'Albumin', 'B/C ratio', 'BMI', 'BUN', 'Basophil', 'CPK', 'Creatinine', 'D.Bilirubin', 'Eosinophil', 'GOT(AST)', 'GPT(ALT)', 'Globulin', 'HDL', 'Hct', 'Hgb', 'I.Bilirubin', 'LDH', 'LDL', 'Lymphocyte', 'MCH', 'MCHC', 'MCV', 'MPV', 'Monocyte', 'PCT', 'PDW', 'PH', 'Platelet', 'RBC', 'RDW', 'SG', 'T.Bilirubin', 'T.Cholesterol', 'T.Protein', 'TSH', 'Triglyceride', 'Uric acid', 'WBC', 'e-GFR', 'r-GTP', '공복혈당', '나이', '비만도', '신장', '체중', '허리둘레', '혈압(수축기)', '혈압(이완기)', 'gender_여자', 'Bilirubin_음성', 'Blood_음성', 'Glucose_음성', 'Keton_음성', 'Leukocyte_음성', 'Nitrite_음성', 'Protein_음성', 'Urobilinogen_음성', 'HBs-Ag_음성', 'HBs-Ab_음성']

── Pre-diabetes (62개) ──
['A/G ratio', 'AFP', 'ALP', 'Albumin', 'B/C ratio', 'BMI', 'BUN', 'Basophil', 'CPK', 'Creatinine', 'D.Bilirubin', 'Eosinophil', 'GOT(AST)', 'GPT(ALT)', 'Globulin', 'HDL', 'Hct', 'Hgb', 'I.Bilirubin', 'LDH', 'LDL', 'Lymphocyte', 'MCH', 'MCHC', 'MCV', 'MPV', 'Monocyte', 'PCT', 'PDW', 'PH', 'Platelet', 'RBC',

---
## 1. Diabetes (당뇨병전단계 → 당뇨병)

In [ ]:
diabetes_config = MLConfig(
    dataset_path          = ROOT / "outputs" / "260526_preprocessing" / "diabetes_dataset.xlsx",
    vif_features_path     = ROOT / "outputs" / "260527_statistic_analysis" / "diabetes" / "vif_survived.csv",
    output_dir            = ROOT / "outputs" / "260528_ML_training" / "diabetes",
    label_col             = "label",
    n_outer_folds         = N_OUTER_FOLDS,
    n_inner_folds         = N_INNER_FOLDS,
    knn_n_neighbors       = KNN_N_NEIGHBORS,
    random_state          = RANDOM_STATE,
    models                = MODELS,
    scale_features        = SCALE_FEATURES,
    selected_features     = DIABETES_FEATURES,
    tuner                 = TUNER,
    n_bayes_iter          = N_BAYES_ITER,
    imputation_strategy   = IMPUTATION_STRATEGY,
    imputation_fill_value = IMPUTATION_FILL_VALUE,
)

diabetes_pipeline = MLPipeline(diabetes_config)
diabetes_results  = diabetes_pipeline.run()

ML Pipeline: diabetes_dataset.xlsx
Output dir : /data2/mason/prediabetes_diabetes/outputs/260528_ML_training/diabetes
Imputation : mean
Models     : ['svm', 'lr', 'rf', 'xgb', 'lgbm', 'gb', 'et']
Tuner      : BayesSearchCV (n_iter=30)
피처 소스: 직접 지정 (64개)
X shape: (739, 62), y distribution: {0: 710, 1: 29}
선택된 피처 (62개): ['A/G ratio', 'AFP', 'ALP', 'Albumin', 'B/C ratio', 'BMI', 'BUN', 'Basophil', 'CPK', 'Creatinine', 'D.Bilirubin', 'Eosinophil', 'GOT(AST)', 'GPT(ALT)', 'Globulin', 'HDL', 'Hct', 'Hgb', 'I.Bilirubin', 'LDH', 'LDL', 'Lymphocyte', 'MCH', 'MCHC', 'MCV', 'MPV', 'Monocyte', 'PCT', 'PDW', 'PH', 'Platelet', 'RBC', 'RDW', 'SG', 'T.Bilirubin', 'T.Cholesterol', 'T.Protein', 'TSH', 'Triglyceride', 'Uric acid', 'WBC', 'e-GFR', 'r-GTP', '공복혈당', '나이', '비만도', '신장', '체중', '허리둘레', '혈압(수축기)', '혈압(이완기)', 'gender_여자', 'Bilirubin_음성', 'Blood_음성', 'Glucose_음성', 'Keton_음성', 'Leukocyte_음성', 'Nitrite_음성', 'Protein_음성', 'Urobilinogen_음성', 'HBs-Ag_음성', 'HBs-Ab_음성']

=================================

/data2/mason/prediabetes_diabetes/core/ml_trainer.py:298: UserWarning: VIF 피처 중 데이터에 없는 컬럼: ['백혈구(소변현미경)', '적혈구(소변현미경)']
  X = self._select_features(df, features)


  [SVM] AUC=0.8838  PR-AUC=0.3114  F1=0.2500  Acc=0.8784  Sens=0.5000  Spec=0.8944
  [LR ] AUC=0.8885  PR-AUC=0.4068  F1=0.4348  Acc=0.9122  Sens=0.8333  Spec=0.9155
  [RF ] AUC=0.8169  PR-AUC=0.1763  F1=0.0000  Acc=0.9595  Sens=0.0000  Spec=1.0000


### 1-1. Fold별 상세 지표

In [ ]:
import pandas as pd
from IPython.display import display

for model_name, result in diabetes_results.items():
    print(f"\n{'─'*50}")
    print(f"[Diabetes] {model_name.upper()} — Fold 상세")
    display(result.metrics_df().set_index("fold"))

print("\n[Diabetes] 전체 요약")
diabetes_pipeline.exporter.print_summary_table(diabetes_results)


──────────────────────────────────────────────────
[Diabetes] SVM — Fold 상세


,model,accuracy,sensitivity,specificity,precision,f1,auc,pr_auc
fold,,,,,,,,
1,svm,0.806818,0.692308,0.812749,0.160714,0.260870,0.850138,0.248101
2,svm,0.821293,0.500000,0.836653,0.127660,0.203390,0.818061,0.265378
3,svm,0.904943,0.166667,0.940239,0.117647,0.137931,0.776892,0.154052
4,svm,0.794677,0.461538,0.812000,0.113208,0.181818,0.741846,0.166342
5,svm,0.798479,0.461538,0.816000,0.115385,0.184615,0.670769,0.115286



──────────────────────────────────────────────────
[Diabetes] LR — Fold 상세


,model,accuracy,sensitivity,specificity,precision,f1,auc,pr_auc
fold,,,,,,,,
1,lr,0.795455,0.692308,0.800797,0.152542,0.250000,0.832363,0.300287
2,lr,0.836502,0.583333,0.848606,0.155556,0.245614,0.815073,0.234084
3,lr,0.870722,0.333333,0.896414,0.133333,0.190476,0.755312,0.206834
4,lr,0.821293,0.461538,0.840000,0.130435,0.203390,0.743692,0.239774
5,lr,0.783270,0.384615,0.804000,0.092593,0.149254,0.692615,0.116664



──────────────────────────────────────────────────
[Diabetes] RF — Fold 상세


...

,model,accuracy,sensitivity,specificity,precision,f1,auc,pr_auc
fold,,,,,,,,
1,gb,0.939394,0.076923,0.984064,0.200000,0.111111,0.719890,0.149198
2,gb,0.931559,0.083333,0.972112,0.125000,0.100000,0.776560,0.135802
3,gb,0.931559,0.166667,0.968127,0.200000,0.181818,0.741036,0.130570
4,gb,0.851711,0.153846,0.888000,0.066667,0.093023,0.585692,0.082342
5,gb,0.920152,0.000000,0.968000,0.000000,0.000000,0.673846,0.123746



──────────────────────────────────────────────────
[Diabetes] ET — Fold 상세


,model,accuracy,sensitivity,specificity,precision,f1,auc,pr_auc
fold,,,,,,,,
1,et,0.867424,0.461538,0.888446,0.176471,0.255319,0.860558,0.330143
2,et,0.863118,0.416667,0.884462,0.147059,0.217391,0.824037,0.150719
3,et,0.855513,0.333333,0.880478,0.117647,0.173913,0.766932,0.176833
4,et,0.893536,0.230769,0.928000,0.142857,0.176471,0.736000,0.159255
5,et,0.825095,0.461538,0.844000,0.133333,0.206897,0.600000,0.132400



[Diabetes] 전체 요약


,accuracy,sensitivity,specificity,precision,f1,auc,pr_auc
model,,,,,,,
SVM,0.8252 ± 0.0457,0.4564 ± 0.1881,0.8435 ± 0.0550,0.1269 ± 0.0197,0.1937 ± 0.0446,0.7715 ± 0.0697,0.1898 ± 0.0642
LR,0.8214 ± 0.0346,0.4910 ± 0.1467,0.8380 ± 0.0390,0.1329 ± 0.0251,0.2077 ± 0.0417,0.7678 ± 0.0566,0.2195 ± 0.0669
RF,0.9194 ± 0.0274,0.1423 ± 0.0638,0.9585 ± 0.0304,0.1733 ± 0.0789,0.1451 ± 0.0495,0.7352 ± 0.0891,0.1682 ± 0.0541
XGB,0.9005 ± 0.0358,0.2628 ± 0.2135,0.9321 ± 0.0456,0.1755 ± 0.1189,0.1762 ± 0.1121,0.7202 ± 0.0244,0.1687 ± 0.0580
LGBM,0.8799 ± 0.0303,0.4564 ± 0.1628,0.9010 ± 0.0371,0.1875 ± 0.0379,0.2608 ± 0.0675,0.7338 ± 0.0644,0.1902 ± 0.0650
GB,0.9149 ± 0.0360,0.0962 ± 0.0672,0.9561 ± 0.0386,0.1183 ± 0.0867,0.0972 ± 0.0649,0.6994 ± 0.0736,0.1243 ± 0.0253
ET,0.8609 ± 0.0246,0.3808 ± 0.0989,0.8851 ± 0.0298,0.1435 ± 0.0216,0.2060 ± 0.0334,0.7575 ± 0.1005,0.1899 ± 0.0800


---
## 2. Pre-diabetes (정상 → 당뇨병전단계)

In [ ]:
prediabetes_config = MLConfig(
    dataset_path          = ROOT / "outputs" / "260526_preprocessing" / "pre_diabetes_dataset.xlsx",
    vif_features_path     = ROOT / "outputs" / "260527_statistic_analysis" / "pre_diabetes" / "vif_survived.csv",
    output_dir            = ROOT / "outputs" / "260528_ML_training" / "pre_diabetes",
    label_col             = "label",
    n_outer_folds         = N_OUTER_FOLDS,
    n_inner_folds         = N_INNER_FOLDS,
    knn_n_neighbors       = KNN_N_NEIGHBORS,
    random_state          = RANDOM_STATE,
    models                = MODELS,
    scale_features        = SCALE_FEATURES,
    selected_features     = PREDIABETES_FEATURES,
    tuner                 = TUNER,
    n_bayes_iter          = N_BAYES_ITER,
    imputation_strategy   = IMPUTATION_STRATEGY,
    imputation_fill_value = IMPUTATION_FILL_VALUE,
)

prediabetes_pipeline = MLPipeline(prediabetes_config)
prediabetes_results  = prediabetes_pipeline.run()

ML Pipeline: pre_diabetes_dataset.xlsx
Output dir : /data2/mason/prediabetes_diabetes/outputs/260528_ML_training/pre_diabetes
KNN k      : 5
Models     : ['svm', 'lr', 'rf', 'xgb', 'lgbm', 'gb', 'et']
Tuner      : BayesSearchCV (n_iter=30)
피처 소스: 직접 지정 (57개)
X shape: (2771, 57), y distribution: {0: 2167, 1: 604}
선택된 피처 (57개): ['A/G ratio', 'ALP', 'Albumin', 'B/C ratio', 'BMI', 'BUN', 'Basophil', 'Creatinine', 'D.Bilirubin', 'Eosinophil', 'GOT(AST)', 'GPT(ALT)', 'Globulin', 'HDL', 'Hct', 'Hgb', 'LDL', 'Lymphocyte', 'MCH', 'MCHC', 'MCV', 'MPV', 'Monocyte', 'PDW', 'PH', 'Platelet', 'RBC', 'RDW', 'SG', 'T.Bilirubin', 'T.Cholesterol', 'T.Protein', 'TSH', 'Triglyceride', 'Uric acid', 'WBC', 'e-GFR', 'r-GTP', '공복혈당', '나이', '비만도', '신장', '체중', '허리둘레', '혈압(수축기)', '혈압(이완기)', 'gender_여자', 'Bilirubin_음성', 'Blood_음성', 'Glucose_음성', 'Keton_음성', 'Leukocyte_음성', 'Nitrite_음성', 'Protein_음성', 'Urobilinogen_음성', 'HBs-Ag_음성', 'HBs-Ab_음성']

============================================================  Fold 1

  [SVM] AUC=0.7056  PR-AUC=0.3619  F1=0.4562  Acc=0.6306  Sens=0.7107  Spec=0.6083
  [LR ] AUC=0.7168  PR-AUC=0.3711  F1=0.4632  Acc=0.6450  Sens=0.7025  Spec=0.6290
  [RF ] AUC=0.7130  PR-AUC=0.3726  F1=0.4531  Acc=0.6216  Sens=0.7190  Spec=0.5945
  [XGB] AUC=0.7288  PR-AUC=0.4022  F1=0.4615  Acc=0.6721  Sens=0.6446  Spec=0.6797


/data2/mason/prediabetes_diabetes/.venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/data2/mason/prediabetes_diabetes/.venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  [LGBM] AUC=0.6901  PR-AUC=0.3617  F1=0.4414  Acc=0.6306  Sens=0.6694  Spec=0.6198
  [GB ] AUC=0.7034  PR-AUC=0.3532  F1=0.3761  Acc=0.7550  Sens=0.3388  Spec=0.8710
  [ET ] AUC=0.7147  PR-AUC=0.3998  F1=0.4516  Acc=0.6324  Sens=0.6942  Spec=0.6152

============================================================  Fold 2
  [SVM] AUC=0.7014  PR-AUC=0.3672  F1=0.4350  Acc=0.6155  Sens=0.6833  Spec=0.5968
  [LR ] AUC=0.7010  PR-AUC=0.3649  F1=0.4470  Acc=0.6516  Sens=0.6500  Spec=0.6521
  [RF ] AUC=0.6915  PR-AUC=0.3822  F1=0.4501  Acc=0.6516  Sens=0.6583  Spec=0.6498
  [XGB] AUC=0.7027  PR-AUC=0.3905  F1=0.4323  Acc=0.6444  Sens=0.6250  Spec=0.6498
  [LGBM] AUC=0.7059  PR-AUC=0.3697  F1=0.4398  Acc=0.6643  Sens=0.6083  Spec=0.6797


/data2/mason/prediabetes_diabetes/.venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/data2/mason/prediabetes_diabetes/.venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


...

/data2/mason/prediabetes_diabetes/.venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/data2/mason/prediabetes_diabetes/.venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  [LGBM] AUC=0.6855  PR-AUC=0.3246  F1=0.4259  Acc=0.6155  Sens=0.6529  Spec=0.6051
  [GB ] AUC=0.5936  PR-AUC=0.2745  F1=0.2342  Acc=0.6931  Sens=0.2149  Spec=0.8268
  [ET ] AUC=0.6927  PR-AUC=0.3728  F1=0.4534  Acc=0.6083  Sens=0.7438  Spec=0.5704

============================================================  Fold 5
  [SVM] AUC=0.7070  PR-AUC=0.3785  F1=0.4706  Acc=0.6426  Sens=0.7273  Spec=0.6189
  [LR ] AUC=0.7100  PR-AUC=0.3770  F1=0.4593  Acc=0.6643  Sens=0.6529  Spec=0.6674
  [RF ] AUC=0.6630  PR-AUC=0.3537  F1=0.4256  Acc=0.5957  Sens=0.6860  Spec=0.5704
  [XGB] AUC=0.7068  PR-AUC=0.4099  F1=0.4743  Acc=0.6679  Sens=0.6860  Spec=0.6628


/data2/mason/prediabetes_diabetes/.venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/data2/mason/prediabetes_diabetes/.venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  [LGBM] AUC=0.6943  PR-AUC=0.3947  F1=0.4419  Acc=0.6534  Sens=0.6281  Spec=0.6605
  [GB ] AUC=0.6774  PR-AUC=0.3352  F1=0.2812  Acc=0.7509  Sens=0.2231  Spec=0.8984
  [ET ] AUC=0.6719  PR-AUC=0.3702  F1=0.4305  Acc=0.6227  Sens=0.6529  Spec=0.6143
Saved metrics: /data2/mason/prediabetes_diabetes/outputs/260528_ML_training/pre_diabetes/cv_metrics.csv
Saved summary: /data2/mason/prediabetes_diabetes/outputs/260528_ML_training/pre_diabetes/cv_summary.csv
Saved confusion matrix: /data2/mason/prediabetes_diabetes/outputs/260528_ML_training/pre_diabetes/cm_svm.png
Saved confusion matrix: /data2/mason/prediabetes_diabetes/outputs/260528_ML_training/pre_diabetes/cm_lr.png
Saved confusion matrix: /data2/mason/prediabetes_diabetes/outputs/260528_ML_training/pre_diabetes/cm_rf.png
Saved confusion matrix: /data2/mason/prediabetes_diabetes/outputs/260528_ML_training/pre_diabetes/cm_xgb.png
Saved confusion matrix: /data2/mason/prediabetes_diabetes/outputs/260528_ML_training/pre_diabetes/cm_lgbm.pn

,accuracy,sensitivity,specificity,precision,f1,auc,pr_auc
model,,,,,,,
SVM,0.6214 ± 0.0148,0.7069 ± 0.0160,0.5976 ± 0.0163,0.3289 ± 0.0127,0.4489 ± 0.0146,0.6965 ± 0.0114,0.3704 ± 0.0068
LR,0.6416 ± 0.0202,0.6622 ± 0.0233,0.6359 ± 0.0270,0.3369 ± 0.0156,0.4464 ± 0.0156,0.6984 ± 0.0159,0.3689 ± 0.0056
RF,0.6214 ± 0.0295,0.6689 ± 0.0338,0.6082 ± 0.0408,0.3234 ± 0.0199,0.4355 ± 0.0195,0.6809 ± 0.0222,0.3517 ± 0.0277
XGB,0.6434 ± 0.0268,0.6638 ± 0.0342,0.6377 ± 0.0380,0.3391 ± 0.0207,0.4484 ± 0.0201,0.7011 ± 0.0203,0.3785 ± 0.0335
LGBM,0.6330 ± 0.0261,0.6473 ± 0.0289,0.6290 ± 0.0408,0.3282 ± 0.0148,0.4350 ± 0.0083,0.6897 ± 0.0121,0.3566 ± 0.0286
GB,0.7416 ± 0.0272,0.2649 ± 0.0509,0.8745 ± 0.0287,0.3740 ± 0.0669,0.3089 ± 0.0542,0.6495 ± 0.0426,0.3284 ± 0.0309
ET,0.6149 ± 0.0147,0.6821 ± 0.0415,0.5962 ± 0.0198,0.3201 ± 0.0134,0.4356 ± 0.0191,0.6838 ± 0.0221,0.3715 ± 0.0225


### 2-1. Fold별 상세 지표

In [ ]:
for model_name, result in prediabetes_results.items():
    print(f"\n{'─'*50}")
    print(f"[Pre-diabetes] {model_name.upper()} — Fold 상세")
    display(result.metrics_df().set_index("fold"))

print("\n[Pre-diabetes] 전체 요약")
prediabetes_pipeline.exporter.print_summary_table(prediabetes_results)


──────────────────────────────────────────────────
[Pre-diabetes] SVM — Fold 상세


,model,accuracy,sensitivity,specificity,precision,f1,auc,pr_auc
fold,,,,,,,,
1,svm,0.630631,0.710744,0.608295,0.335938,0.456233,0.705583,0.361916
2,svm,0.615523,0.683333,0.596774,0.319066,0.435013,0.701440,0.367182
3,svm,0.611913,0.710744,0.584296,0.323308,0.444444,0.683755,0.376257
4,svm,0.606498,0.702479,0.579677,0.318352,0.438144,0.684748,0.368171
5,svm,0.642599,0.727273,0.618938,0.347826,0.470588,0.707041,0.378491



──────────────────────────────────────────────────
[Pre-diabetes] LR — Fold 상세


,model,accuracy,sensitivity,specificity,precision,f1,auc,pr_auc
fold,,,,,,,,
1,lr,0.645045,0.702479,0.629032,0.345528,0.463215,0.716799,0.371138
2,lr,0.651625,0.650000,0.652074,0.340611,0.446991,0.701018,0.364904
3,lr,0.637184,0.644628,0.635104,0.330508,0.436975,0.683603,0.368522
4,lr,0.610108,0.661157,0.595843,0.313725,0.425532,0.680587,0.362742
5,lr,0.664260,0.652893,0.667436,0.354260,0.459302,0.710019,0.377016



──────────────────────────────────────────────────
[Pre-diabetes] RF — Fold 상세


...

,model,accuracy,sensitivity,specificity,precision,f1,auc,pr_auc
fold,,,,,,,,
1,gb,0.754955,0.338843,0.870968,0.422680,0.376147,0.703431,0.353168
2,gb,0.756318,0.258333,0.894009,0.402597,0.314721,0.628687,0.338896
3,gb,0.752708,0.289256,0.882217,0.406977,0.338164,0.644075,0.340251
4,gb,0.693141,0.214876,0.826790,0.257426,0.234234,0.593648,0.274521
5,gb,0.750903,0.223140,0.898383,0.380282,0.281250,0.677438,0.335236



──────────────────────────────────────────────────
[Pre-diabetes] ET — Fold 상세


,model,accuracy,sensitivity,specificity,precision,f1,auc,pr_auc
fold,,,,,,,,
1,et,0.632432,0.694215,0.615207,0.334661,0.451613,0.714743,0.399803
2,et,0.617329,0.683333,0.599078,0.320312,0.436170,0.683794,0.377517
3,et,0.593863,0.636364,0.581986,0.298450,0.406332,0.656023,0.336977
4,et,0.608303,0.743802,0.570439,0.326087,0.453401,0.692688,0.372802
5,et,0.622744,0.652893,0.614319,0.321138,0.430518,0.671865,0.370188



[Pre-diabetes] 전체 요약


,accuracy,sensitivity,specificity,precision,f1,auc,pr_auc
model,,,,,,,
SVM,0.6214 ± 0.0148,0.7069 ± 0.0160,0.5976 ± 0.0163,0.3289 ± 0.0127,0.4489 ± 0.0146,0.6965 ± 0.0114,0.3704 ± 0.0068
LR,0.6416 ± 0.0202,0.6622 ± 0.0233,0.6359 ± 0.0270,0.3369 ± 0.0156,0.4464 ± 0.0156,0.6984 ± 0.0159,0.3689 ± 0.0056
RF,0.6214 ± 0.0295,0.6689 ± 0.0338,0.6082 ± 0.0408,0.3234 ± 0.0199,0.4355 ± 0.0195,0.6809 ± 0.0222,0.3517 ± 0.0277
XGB,0.6434 ± 0.0268,0.6638 ± 0.0342,0.6377 ± 0.0380,0.3391 ± 0.0207,0.4484 ± 0.0201,0.7011 ± 0.0203,0.3785 ± 0.0335
LGBM,0.6330 ± 0.0261,0.6473 ± 0.0289,0.6290 ± 0.0408,0.3282 ± 0.0148,0.4350 ± 0.0083,0.6897 ± 0.0121,0.3566 ± 0.0286
GB,0.7416 ± 0.0272,0.2649 ± 0.0509,0.8745 ± 0.0287,0.3740 ± 0.0669,0.3089 ± 0.0542,0.6495 ± 0.0426,0.3284 ± 0.0309
ET,0.6149 ± 0.0147,0.6821 ± 0.0415,0.5962 ± 0.0198,0.3201 ± 0.0134,0.4356 ± 0.0191,0.6838 ± 0.0221,0.3715 ± 0.0225
